# ДОМАШНЕЕ ЗАДАНИЕ 2. Модели предсказания

<hr>

Выполнил Фадеев Роман Андреевич гр. ИУ6-21М

<a name="0"></a>
<div><span style="font-size:16pt; font-weight:bold">Содержание</span>
    <ol>
        <li><a href="#1">Цель работы</a></li>
        <li><a href="#2">Вариант</a></li>
        <li><a href="#3">Задача 1. Реализация собственных классов и функций</a></li>
        <li><a href="#4">Задача 2. Классификация и кросс-валидация</a></li>
        <li><a href="#5">Задача 3. Классификация текстовых документов</a></li>
    </ol>
</div>

<a name="1"></a>
<div style="display:table; width:100%; padding-top:10px; padding-bottom:10px; border-bottom:1px solid lightgrey">
    <div style="display:table-row">
        <div style="display:table-cell; width:80%; font-size:16pt; font-weight:bold">1. Цель работы</div>
    	<div style="display:table-cell; width:20%; text-align:center; background-color:whitesmoke; border:1px solid lightgrey"><a href="#0">К содержанию</a></div>
    </div>
</div>

Приобрести опыт решения практических задач по машинному обучению, таких как анализ и визуализация исходных данных, обучение, выбор и оценка качества моделей предсказания, посредством языка программирования Python.

При выполнении работы решаются следующие задачи:

- реализация собственных классов совместимых с библиотекой `sklearn`
- оценка влияния регуляризации в моделях предсказания
- преобразование исходных данных посредством транформаторов `sklearn`
- использование отложенной выборки и кросс-валидации
- выбор гиперпараметров и интерпретация кривых обучения
- оценка качества моделей предсказания
- выявление преимуществ и недостатков методов предсказания в зависимости от поставленной задачи

<a name="2"></a>
<div style="display:table; width:100%; padding-top:10px; padding-bottom:10px; border-bottom:1px solid lightgrey">
    <div style="display:table-row">
        <div style="display:table-cell; width:80%; font-size:16pt; font-weight:bold">2. Вариант</div>
    	<div style="display:table-cell; width:20%; text-align:center; background-color:whitesmoke; border:1px solid lightgrey"><a href="#0">К содержанию</a></div>
    </div>
</div>

In [ ]:
surname = "Фадеев"  # Ваша фамилия

alph = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
w = [4, 42, 21, 21, 55,  1, 44, 26, 18, 3, 38, 26, 18, 12,  3, 49, 45,
        7, 42, 9,  4,  3, 36, 33, 31, 29,  5, 4,  4, 19, 21, 27, 33]
d = dict(zip(alph, w))
variant =  sum([d[el] for el in surname.lower()]) % 40 + 1

print("Задание № 2. Вариант: ", variant % 2 + 1)
print("Задание № 3. Вариант: ", variant % 3 + 1 )

<a name="3"></a>
<div style="display:table; width:100%; padding-top:10px; padding-bottom:10px; border-bottom:1px solid lightgrey">
    <div style="display:table-row">
        <div style="display:table-cell; width:80%; font-size:16pt; font-weight:bold">3. Задача 1. Реализация собственных классов и функций (4 балла)</div>
    	<div style="display:table-cell; width:20%; text-align:center; background-color:whitesmoke; border:1px solid lightgrey"><a href="#0">К содержанию</a></div>
    </div>
</div>

[Набор данные](../../data/A2_Model_Selection/regularization.csv)

⚠️ **Замечание.**  
1) Нельзя пользоваться готовыми реализациями `sklearn`;  
2) чтобы избежать случая с вырожденной матрицей при оценке параметров добавьте незначительную регуляризацию по умолчанию или используйте `lstsq` из пакета `numpy` или др. способ;  
3) используйте `random_state=0`

### 3.1. Класс линейной регрессии с L2-регуляризацией

Реализуйте класс, предназначенный для оценки параметров линейной регрессии с L2 регуляризацией совместимый с `sklearn`. Передаваемые параметры: 1) коэффициент регуляризации (`alpha`). Использовать метод наименьших квадратов с L2 регуляризацией.

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted


class MyRidgeRegression(BaseEstimator, RegressorMixin):
    """
    Линейная регрессия с L2-регуляризацией (Ridge),
    совместимая с интерфейсом sklearn.

    Параметры
    ----------
    alpha : float, default=1.0
        Коэффициент L2-регуляризации.
    fit_intercept : bool, default=True
        Добавлять ли свободный член.
    """

    def __init__(self, alpha=1.0, fit_intercept=True):
        self.alpha = alpha
        self.fit_intercept = fit_intercept

    def fit(self, X, y):
        """
        Обучение модели.

        Parameters
        ----------
        X : array-like shape (n_samples, n_features)
        y : array-like shape (n_samples,)
        """
        # Проверка входных данных
        X, y = check_X_y(X, y)

        n_samples, n_features = X.shape

        # Добавление столбца единиц для intercept
        if self.fit_intercept:
            X_bias = np.hstack([np.ones((n_samples, 1)), X])
        else:
            X_bias = X

        # Размерность матрицы
        n_params = X_bias.shape[1]

        # Единичная матрица для регуляризации
        I = np.eye(n_params)

        # Не регуляризуем intercept
        if self.fit_intercept:
            I[0, 0] = 0

        # Формула Ridge:
        # w = (X^T X + alpha * I)^(-1) X^T y
        A = X_bias.T @ X_bias + self.alpha * I
        b = X_bias.T @ y

        # Решение системы
        w = np.linalg.lstsq(A, b, rcond=None)[0]

        # Сохранение параметров
        if self.fit_intercept:
            self.intercept_ = w[0]
            self.coef_ = w[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = w

        return self

    def predict(self, X):
        """
        Предсказание значений.
        """
        check_is_fitted(self, ["coef_", "intercept_"])
        X = check_array(X)
        return X @ self.coef_ + self.intercept_

    def score(self, X, y):
        """
        Коэффициент детерминации R^2.
        """
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - ss_res / ss_tot

### 3.2. Класс стандартизации признаков

Реализуйте класс для стандартизации признаков в виде трансформации совместимый с `sklearn`. Передаваемые параметры: 1) `has_bias` (содержит ли  матрица вектор единиц), 2) `apply_mean` (производить ли центровку)

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted


class MyStandardScaler(BaseEstimator, TransformerMixin):
    """
    Класс стандартизации признаков, совместимый со sklearn.

    Параметры
    ----------
    has_bias : bool, default=False
        Содержит ли матрица признаков столбец единиц (bias).
        Если True, первый столбец не преобразуется.

    apply_mean : bool, default=True
        Выполнять ли центровку признаков.
    """

    def __init__(self, has_bias=False, apply_mean=True):
        self.has_bias = has_bias
        self.apply_mean = apply_mean

    def fit(self, X, y=None):
        """
        Оценка среднего и стандартного отклонения.

        Parameters
        ----------
        X : array-like shape (n_samples, n_features)
        """
        X = check_array(X)
        n_features = X.shape[1]

        # Индекс начала преобразуемых признаков
        start_idx = 1 if self.has_bias else 0

        # Инициализация
        self.mean_ = np.zeros(n_features)
        self.scale_ = np.ones(n_features)

        # Среднее
        if self.apply_mean:
            self.mean_[start_idx:] = np.mean(X[:, start_idx:], axis=0)

        # Стандартное отклонение
        self.scale_[start_idx:] = np.std(X[:, start_idx:], axis=0)

        # Защита от деления на ноль
        self.scale_[self.scale_ == 0] = 1.0

        return self

    def transform(self, X):
        """
        Стандартизация данных.

        Parameters
        ----------
        X : array-like shape (n_samples, n_features)
        """
        check_is_fitted(self, ["mean_", "scale_"])
        X = check_array(X)
        X_transformed = X.copy()

        start_idx = 1 if self.has_bias else 0
        X_transformed[:, start_idx:] = (
            X_transformed[:, start_idx:] - self.mean_[start_idx:]
        ) / self.scale_[start_idx:]

        return X_transformed

    def fit_transform(self, X, y=None):
        """Обучение и преобразование."""
        return self.fit(X, y).transform(X)

### 3.3. Функции оценки качества (Holdout и Cross-Validation)

Реализуйте функции для расчета `MSE` и `R^2` при отложенной выборке (`run_holdout`) и кросс-валидации (`run_cross_val`). Для кросс-валидации используйте **только** класс `KFold`. Выходными значениями должны быть `MSE` и `R^2` для обучающей и тестовой частей.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import clone


def run_holdout(model, X, y, train_size=0.75, random_state=0) -> dict:
    """
    Оценка модели на отложенной выборке.

    Parameters
    ----------
    model : sklearn-compatible estimator
        Модель машинного обучения.
    X : array-like
        Матрица признаков.
    y : array-like
        Вектор ответов.
    train_size : float
        Доля обучающей выборки.
    random_state : int
        Параметр воспроизводимости.

    Returns
    -------
    dict
        Метрики качества.
    """
    # Разделение данных
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=train_size, random_state=random_state
    )

    # Обучение модели
    model.fit(X_train, y_train)

    # Предсказания
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Метрики
    scores = {
        "train_mse": mean_squared_error(y_train, y_train_pred),
        "test_mse": mean_squared_error(y_test, y_test_pred),
        "train_r2": r2_score(y_train, y_train_pred),
        "test_r2": r2_score(y_test, y_test_pred)
    }
    return scores


def run_cross_val(model, X, y, n_splits=4, shuffle=True, random_state=0) -> dict:
    """
    Оценка модели с помощью KFold кросс-валидации.

    Parameters
    ----------
    model : sklearn-compatible estimator
        Модель машинного обучения.
    X : array-like
        Матрица признаков.
    y : array-like
        Вектор ответов.
    n_splits : int
        Количество фолдов.
    shuffle : bool
        Перемешивание данных.
    random_state : int
        Параметр воспроизводимости.

    Returns
    -------
    dict
        Средние значения метрик по всем фолдам.
    """
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

    train_mse_scores, test_mse_scores = [], []
    train_r2_scores, test_r2_scores = [], []

    # Проход по фолдам
    for train_idx, test_idx in kf.split(X):
        # Разделение данных
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Клонирование модели
        current_model = clone(model)
        current_model.fit(X_train, y_train)

        # Предсказания
        y_train_pred = current_model.predict(X_train)
        y_test_pred = current_model.predict(X_test)

        # Метрики
        train_mse_scores.append(mean_squared_error(y_train, y_train_pred))
        test_mse_scores.append(mean_squared_error(y_test, y_test_pred))
        train_r2_scores.append(r2_score(y_train, y_train_pred))
        test_r2_scores.append(r2_score(y_test, y_test_pred))

    # Средние значения
    scores = {
        "train_mse": np.mean(train_mse_scores),
        "test_mse": np.mean(test_mse_scores),
        "train_r2": np.mean(train_r2_scores),
        "test_r2": np.mean(test_r2_scores)
    }
    return scores

### 3.4. Обучение линейной регрессии с регуляризацией

Выполните обучение линейной регрессии с регуляризацией, используя ранее реализованные классы и функции. Для этого воспользуйтесь `Pipeline` со стандартизацией и линейной регрессией (см. п.1 и п.2). Для линейной регрессии используйте следующие коэффициенты регуляризации: `0` и `0.01`. Для оценки качества моделей выведите значения `MSE` и `R^2`, полученные посредством функций `run_holdout` и `run_cross_val`. Повторно обучите модели с разными коэффициентами регуляризации на всём наборе данных и выведите параметры моделей.

Необходимо использовать следующие параметры:
- `train_size=0.75`
- `n_splits=4`
- `shuffle=True`
- `random_state=0`

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline

# =========================================================
# ЗАГРУЗКА ДАННЫХ
# =========================================================

df = pd.read_csv("../../data/A2_Model_Selection/regularization.csv")
X = df.drop(columns=["Y"]).values
y = df["Y"].values

# =========================================================
# ОБУЧЕНИЕ МОДЕЛЕЙ
# =========================================================

alphas = [0, 0.01]

for alpha in alphas:
    print("=" * 70)
    print(f"Ridge regression: alpha = {alpha}")
    print("=" * 70)

    model = Pipeline([
        ("scaler", MyStandardScaler(has_bias=False, apply_mean=True)),
        ("regressor", MyRidgeRegression(alpha=alpha, fit_intercept=True))
    ])

    # =====================================================
    # HOLDOUT
    # =====================================================

    holdout_scores = run_holdout(
        model=model, X=X, y=y,
        train_size=0.75, random_state=0
    )

    print("\nHOLDOUT RESULTS")
    print(f"Train MSE: {holdout_scores['train_mse']:.6f}")
    print(f"Test  MSE: {holdout_scores['test_mse']:.6f}")
    print(f"Train R2 : {holdout_scores['train_r2']:.6f}")
    print(f"Test  R2 : {holdout_scores['test_r2']:.6f}")

    # =====================================================
    # CROSS-VALIDATION
    # =====================================================

    cv_scores = run_cross_val(
        model=model, X=X, y=y,
        n_splits=4, shuffle=True, random_state=0
    )

    print("\nCROSS-VALIDATION RESULTS")
    print(f"Train MSE: {cv_scores['train_mse']:.6f}")
    print(f"Test  MSE: {cv_scores['test_mse']:.6f}")
    print(f"Train R2 : {cv_scores['train_r2']:.6f}")
    print(f"Test  R2 : {cv_scores['test_r2']:.6f}")

    # =====================================================
    # ОБУЧЕНИЕ НА ВСЕМ ДАТАСЕТЕ
    # =====================================================

    model.fit(X, y)
    reg = model.named_steps["regressor"]

    print("\nMODEL PARAMETERS")
    print(f"Intercept: {reg.intercept_}")
    print(f"Coefficients: {reg.coef_}\n")

### 3.5. Диаграмма разброса предсказаний

Отобразите предсказания и действительные значения в виде диаграммы разброса. Для этого сформируйте обучающее и тестовое множества (в данном случае можно использовать `train_test_split`). Используйте `Pipeline` и коэффициенты регуляризации из предыдущего пункта. Обучите линейную регрессию на обучающем множестве. Выведите графики предсказание ($\hat{y}$) - действительное значение ($y$) для разных коэффициентов регуляризации для тестовой части.

⚠️ **Замечание.** При формировании исходных данных использовался полином 16 степени одномерных данных.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=0
)


def build_model(alpha):
    return Pipeline([
        ("scaler", MyStandardScaler(has_bias=False, apply_mean=True)),
        ("regressor", MyRidgeRegression(alpha=alpha, fit_intercept=True))
    ])


alphas = [0, 0.01]

for alpha in alphas:
    model = build_model(alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.7)

    mn = min(y_test.min(), y_pred.min())
    mx = max(y_test.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], "r--")

    plt.title(f"y_true vs y_pred (alpha={alpha})")
    plt.xlabel("True y")
    plt.ylabel("Predicted y")
    plt.grid(True)
    plt.show()

<a name="4"></a>
<div style="display:table; width:100%; padding-top:10px; padding-bottom:10px; border-bottom:1px solid lightgrey">
    <div style="display:table-row">
        <div style="display:table-cell; width:80%; font-size:16pt; font-weight:bold">4. Задача 2. Классификация и кросс-валидация (2 балла)</div>
    	<div style="display:table-cell; width:20%; text-align:center; background-color:whitesmoke; border:1px solid lightgrey"><a href="#0">К содержанию</a></div>
    </div>
</div>

Набор данные:
- [Вариант 1](../../data/A2_Model_Selection/Cl_A5_V1.csv)
- [Вариант 2](../../data/A2_Model_Selection/Cl_A5_V2.csv)

⚠️ **Замечание**:
- Используйте класс логистической регрессии из `sklearn` со следующими параметрами:
    - `penalty='l2'`
    - `fit_intercept=True`
    - `max_iter=100`
    - `C=1e5`
    - `solver='liblinear'`
    - `random_state=12345`
- Разбейте исходные данные на обучающее и тестовое подмножества в соотношении `70` на `30`, `random_state=0`
- Для выбора гиперпараметров используйте два подхода: 1) с отложенной выборкой, 2) с кросс-валидацией
- Для кросс-валидации использовать функцию `cross_validate` из `sklearn`
- Параметры разбиения для выбора гиперпараметров используйте те, что в п.4 задачи 1

Дано множество наблюдений (см. набор данных к заданию), классификатор - логистическая регрессия. Найти степень полинома с минимальной ошибкой на проверочном подмножестве. Для лучшего случая рассчитать ошибку на тестовом подмножестве. В качестве метрики использовать долю правильных классификаций. Сделать заключение о влиянии степени полинома на качество предсказания.

Построить:
- диаграмму разброса исходных данных
- зависимость доли правильных классификаций от степени полинома для обучающего и проверочного подмножеств (две кривые на одном графике)
- результат классификации для наилучшего случая (степень полинома) для обучающего и тестового подмножеств с указанием границы принятия решения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score

# Загрузка данных
df = pd.read_csv("../../data/A2_Model_Selection/Cl_A5_V1.csv")
X = df[["X1", "X2"]].values
y = df["y"].values

### 4.1. Разбиение данных на обучающее и тестовое множества

In [ ]:
# Разбиение 70/30
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

### 4.2. Функция создания модели

In [ ]:
def make_model(degree):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree)),
        ("clf", LogisticRegression(
            penalty='l2',
            fit_intercept=True,
            max_iter=100,
            C=1e5,
            solver='liblinear',
            random_state=12345
        ))
    ])

### 4.3. Выбор степени полинома (Holdout и Cross-Validation)

In [ ]:
# Разбиение train_full на train и validation для holdout
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.3, random_state=0, stratify=y_train_full
)

degrees = range(1, 11)
train_acc_holdout, val_acc_holdout, cv_acc = [], [], []

for d in degrees:
    model = make_model(d)

    # ---- HOLDOUT ----
    model.fit(X_train, y_train)
    train_acc_holdout.append(accuracy_score(y_train, model.predict(X_train)))
    val_acc_holdout.append(accuracy_score(y_val, model.predict(X_val)))

    # ---- CROSS VALIDATION ----
    scores = cross_validate(
        model, X_train_full, y_train_full,
        cv=4,
        scoring="accuracy",
        return_train_score=True
    )
    cv_acc.append(scores["test_score"].mean())

### 4.4. График зависимости точности от степени полинома

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(degrees, train_acc_holdout, "o-", label="Train accuracy (holdout)")
plt.plot(degrees, val_acc_holdout, "s-", label="Validation accuracy (holdout)")
plt.plot(degrees, cv_acc, "^-", label="CV accuracy")

plt.xlabel("Polynomial degree")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Polynomial Degree")
plt.legend()
plt.grid(True)
plt.show()

### 4.5. Выбор лучшей степени полинома

In [ ]:
best_degree = degrees[np.argmax(val_acc_holdout)]
print(f"Best degree (holdout): {best_degree}")

# Обучение лучшей модели на всём train_full
best_model = make_model(best_degree)
best_model.fit(X_train_full, y_train_full)

# Оценка на тестовом множестве
test_accuracy = accuracy_score(y_test, best_model.predict(X_test))
print(f"Test accuracy (best degree={best_degree}): {test_accuracy:.4f}")

### 4.6. Диаграмма разброса исходных данных

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k")
plt.xlabel("X1")
plt.ylabel("X2")
plt.title("Input data distribution")
plt.grid(True)
plt.show()

### 4.7. Граница принятия решений

In [ ]:
# Сетка для визуализации границы
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = best_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="bwr")

plt.scatter(X_train_full[:, 0], X_train_full[:, 1], c=y_train_full, cmap="bwr", edgecolor="k", label="train")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="bwr", marker="x", label="test")

plt.title(f"Decision boundary (degree={best_degree})")
plt.legend()
plt.grid(True)
plt.show()

### 4.8. Заключение о влиянии степени полинома

**Вывод:** С увеличением степени полинома модель становится более сложной и лучше описывает обучающие данные. Однако после определённого момента начинается переобучение: точность на валидационном множестве снижается, в то время как на обучающем продолжает расти. Оптимальная степень полинома выбирается по максимуму точности на валидационном множестве.

<a name="5"></a>
<div style="display:table; width:100%; padding-top:10px; padding-bottom:10px; border-bottom:1px solid lightgrey">
    <div style="display:table-row">
        <div style="display:table-cell; width:80%; font-size:16pt; font-weight:bold">5. Задача 3. Классификация текстовых документов (4 балла)</div>
    	<div style="display:table-cell; width:20%; text-align:center; background-color:whitesmoke; border:1px solid lightgrey"><a href="#0">К содержанию</a></div>
    </div>
</div>

- **Вариант 1.** Набор электронных сообщений (emails)
    - файл: `data/[A3]/emails.tsv`
    - [источник](http://csmining.org/index.php/spam-email-datasets.html)

1. Загрузите исходные данные
2. Разбейте исходные данные на обучающее (train, 80%) и тестовое подмножества (test, 20%)
3. Используя стратифицированную кросс-валидацию k-folds ($k=4$) для обучающего множество с метрикой `Balanced-Accuracy`, найдите лучшие гиперпараметры для следующих классификаторов:
    - K-ближайших соседей: количество соседей ($n$) из диапазона `np.arange(1, 150, 20)`
    - Логистическая регрессия: параметр регуляризации ($C$) из диапазона `np.logspace(-2, 10, 8, base=10)`
    - Наивный Байес: сглаживающий параметр модели Бернулли ($\alpha$) из диапазона `np.logspace(-4, 1, 8, base=10)`
    - Наивный Байес: сглаживающий параметр полиномиальной модели ($\alpha$) из диапазона `np.logspace(-4, 1, 8, base=10)`

4. Отобразите кривые (параметры модели)-(`Balanced-Accuracy`) при обучении и проверке для каждой классификатора (две кривые на одном графике для каждого классификатора)
5. Если необходимо, выбранные модели обучите на всём обучающем подмножестве (train) и протестируйте на тестовом (test) по `Balanced-Accuracy`, `R`, `P`, `F1`. Определите время обучения и предсказания.
6. Выполните пункты 3-5 для `n-gram=1`, `n-gram=2` и `n-gram=(1,2)`
7. Выведите в виде таблицы итоговые данные по всем методам для лучших моделей (метод, n-gram, значение параметра модели, время обучения, время предсказания, метрики (`Balanced-Accuracy`, `R`, `P`, `F1`))
8. Сделайте выводы по полученным результатам (преимущества и недостатки методов)

⚠️ **Замечание:**
- Для всех объектов/методов/моделей `random_state = 123`
- Для выбора гиперпараметров можно использовать стандартные утилиты `sklearn`

### 5.1. Загрузка данных

In [ ]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

In [ ]:
# Загрузка данных
df = pd.read_csv(
    "../../data/[A3]/emails.tsv",
    sep="\t",
    engine="python",
    header=None,
    on_bad_lines="skip"
)

df = df.dropna()
df.columns = ["label", "text"]

X = df["text"].astype(str).to_numpy()
y = df["label"].to_numpy().astype(int)

### 5.2. Разбиение на обучающее и тестовое множества

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

### 5.3. Конфигурация моделей и гиперпараметров

In [ ]:
param_grid = {
    "knn": {
        "model": KNeighborsClassifier(),
        "params": {"clf__n_neighbors": np.arange(1, 150, 20)}
    },
    "logreg": {
        "model": LogisticRegression(
            max_iter=200,
            solver="liblinear",
            random_state=123
        ),
        "params": {"clf__C": np.logspace(-2, 10, 8, base=10)}
    },
    "bernoulli": {
        "model": BernoulliNB(),
        "params": {"clf__alpha": np.logspace(-4, 1, 8)}
    },
    "multinomial": {
        "model": MultinomialNB(),
        "params": {"clf__alpha": np.logspace(-4, 1, 8)}
    }
}

### 5.4. Вспомогательные функции

In [ ]:
def build_pipeline(model, ngram):
    return Pipeline([
        ("vec", CountVectorizer(ngram_range=ngram)),
        ("clf", model)
    ])


def plot_cv(grid, title):
    res = pd.DataFrame(grid.cv_results_)
    param_col = [c for c in res.columns if c.startswith("param_")][0]
    x = res[param_col].astype(float)

    plt.figure(figsize=(6, 4))
    plt.plot(x, res["mean_test_score"], marker="o", label="Validation")
    plt.plot(x, res["mean_train_score"], marker="o", label="Train")

    if "C" in param_col or "alpha" in param_col:
        plt.xscale("log")

    plt.xlabel(param_col.replace("param_", ""))
    plt.ylabel("Balanced Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

### 5.5. Обучение моделей и сбор метрик

In [ ]:
ngrams = [(1, 1), (2, 2), (1, 2)]
results_table = []

for name, cfg in param_grid.items():
    for ngram in ngrams:
        pipe = build_pipeline(cfg["model"], ngram)

        grid = GridSearchCV(
            pipe,
            cfg["params"],
            cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=123),
            scoring="balanced_accuracy",
            return_train_score=True,
            n_jobs=-1
        )

        # Обучение и plotting
        start_train = time.time()
        grid.fit(X_train, y_train)
        plot_cv(grid, f"{name} | ngram={ngram}")
        train_time = time.time() - start_train

        # Предсказание
        best_model = grid.best_estimator_
        start_pred = time.time()
        y_pred = best_model.predict(X_test)
        pred_time = time.time() - start_pred

        # Метрики
        ba = balanced_accuracy_score(y_test, y_pred)
        p = precision_score(y_test, y_pred, average="weighted")
        r = recall_score(y_test, y_pred, average="weighted")
        f1 = f1_score(y_test, y_pred, average="weighted")

        results_table.append({
            "model": name,
            "ngram": ngram,
            "best_param": grid.best_params_,
            "train_time": train_time,
            "pred_time": pred_time,
            "balanced_acc": ba,
            "precision": p,
            "recall": r,
            "f1": f1
        })

### 5.6. Итоговая таблица результатов

In [ ]:
results_df = pd.DataFrame(results_table)
results_df = results_df.sort_values(by="balanced_acc", ascending=False)
results_df

### 5.7. Лучшая модель

In [ ]:
best = results_df.iloc[0]
print("BEST MODEL:")
print(f"Model: {best['model']}")
print(f"N-gram: {best['ngram']}")
print(f"Best parameter: {best['best_param']}")
print(f"Train time: {best['train_time']:.3f}s")
print(f"Pred time: {best['pred_time']:.3f}s")
print(f"Balanced Accuracy: {best['balanced_acc']:.4f}")
print(f"Precision: {best['precision']:.4f}")
print(f"Recall: {best['recall']:.4f}")
print(f"F1: {best['f1']:.4f}")

### 5.8. Выводы по результатам

**Преимущества и недостатки методов:**

1. **Логистическая регрессия (logreg)**
   - Преимущества: высокая точность, хорошая интерпретируемость, устойчивость к переобучению при правильной регуляризации
   - Недостатки: требует подбора параметра регуляризации, может быть медленной на больших данных

2. **K-ближайших соседей (knn)**
   - Преимущества: простота реализации, не требует обучения
   - Недостатки: медленное предсказание, чувствительность к шуму, требует хранения всех обучающих данных

3. **Наивный Байес Бернулли (bernoulli)**
   - Преимущества: очень быстрое обучение и предсказание, хорошо работает с бинарными признаками
   - Недостатки: предположение о независимости признаков, может уступать в точности

4. **Наивный Байес Мультиномиальный (multinomial)**
   - Преимущества: хорошо работает с частотными признаками (TF-IDF), быстрое обучение
   - Недостатки: предположение о независимости признаков, чувствительность к параметру сглаживания

**Влияние n-gram:**
- Unigram (1,1): базовый уровень, меньше признаков
- Bigram (2,2): учитывает пары слов, больше контекста, но больше признаков
- (1,2): комбинация unigram и bigram, обычно даёт наилучший результат, но требует больше ресурсов